# Specification Analysis Compliance Calculation

This notebook analyzes selected parametric specifications, calculates compliance from associated measurements, updates spec properties, and uploads an Excel report to the product.


### Imports

Import Python modules for data processing, unit conversion, notebook result recording, and SystemLink API access. The `nisystemlink.clients` package provides the SystemLink clients used by this notebook.


In [ ]:
import io
import json
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import scrapbook as sb
from ni_unit_converter import convert_from_base_unit, convert_to_base_unit
from nisystemlink.clients.core import JupyterHttpConfiguration
from nisystemlink.clients.file import FileClient
from nisystemlink.clients.product import ProductClient
from nisystemlink.clients.product.models import UpdateProductRequest
from nisystemlink.clients.spec import SpecClient
from nisystemlink.clients.spec.models import (
    QuerySpecificationsRequest,
    UpdateSpecificationsRequest,
)
from nisystemlink.clients.testmonitor import TestMonitorClient
from nisystemlink.clients.testmonitor.models import QueryStepsRequest


### Parameters

1. `spec_ids` - IDs of the specifications.
1. `product_id` - ID of the product that contains the specifications and test results.

The parameter cell is tagged `parameters` so Papermill and SystemLink can supply these inputs.


In [ ]:
spec_ids = [""]
product_id = ""
excel_file_name = "Spec_Compliance.xlsx"

### Metadata

The parameter cell metadata defines the Papermill defaults and the SystemLink parameter names and types. If you add or rename an input, update both the code cell and its metadata.


### Constants

In [ ]:
QUERY_SPECS_TAKE_COUNT = 1000
QUERY_STEPS_TAKE_COUNT = 1000


### Data fields and compliance properties

In [ ]:
class ApiBody:
    PRODUCT_IDS = "productIds"
    TAKE = "take"
    TYPE = "type"
    PARAMETRIC = "PARAMETRIC"
    SPECS = "specs"
    CONTINUATION_TOKEN = "continuationToken"
    JSON = "JSON"
    RESPONSE_FORMAT = "responseFormat"
    SPEC_ID = "SpecID"
    STEP_ID = "STEP_ID"
    FILTER = "filter"
    RESULT_FILTER = "resultFilter"
    ORDER_BY = "orderBy"
    FILE_IDS = "fileIds"
    PART_NUMBER = "partNumber"
    DATA = "data"
    PARAMETERS = "parameters"
    ANY = "Any"
    IT = "it"
    PROJECTION = "projection"
    PRODUCTS = "products"
    REPLACE = "replace"
    FALSE = "false"
    WORKSPACE = "workspace"


class ApiResponse:
    TYPE = "type"
    ID = "id"
    SPEC_ID = "specId"
    SPECS = "specs"
    STEPS = "steps"
    PARAMETRIC = "PARAMETRIC"
    PRODUCTS = "products"
    AVAILABLE_FILES = "availableFiles"
    WORKSPACE = "workspace"
    FILE_IDS = "fileIds"
    UPDATED_AT = "updatedAt"
    URI = "uri"
    EMPTY = "Empty"


class Projection:
    INPUTS = "INPUTS"
    DATA = "DATA"
    STEP_ID = "STEP_ID"


class Spec:
    SPECID = "specId"
    CATEGORY = "category"
    BLOCK = "block"
    SYMBOL = "symbol"
    LIMIT = "limit"
    CONDITIONS = "conditions"
    NAME = "name"
    VALUE = "value"
    RANGE = "range"
    MIN = "min"
    MAX = "max"
    STEP = "step"
    TYPICAL = "typical"
    UNIT = "unit"
    DISCRETE = "discrete"
    PROPERTIES = "properties"


class Step:
    INPUTS = "inputs"
    NAME = "name"
    VALUE = "value"
    DATA = "data"
    PARAMETERS = "parameters"
    MEASUREMENT = "measurement"
    SPEC_ID = "SpecID"
    UNITS = "units"


class DataframeHeaders:
    SPEC_DETAILS = "Spec Details"
    SPEC_ID = "SpecID"
    CATEGORY = "Category"
    BLOCK = "Block"
    SYMBOL = "Symbol"
    NAME = "Name"
    MIN = "Min"
    TYPICAL = "Typical"
    MAX = "Max"
    UNIT = "Unit"
    DISCRETE = "Discrete"
    STEP = "Step"
    CONDITION = "Condition"
    COMPLIANCE = "Compliance"
    CP = "Cp"
    CPK = "Cpk"
    STANDARD_DEVIATION = "Standard Deviation"


class ComplianceParameters:
    MIN = "Min"
    MAX = "Max"
    MEAN = "Mean"
    MEDIAN = "Median"
    HEALTH = "Health"
    CP = "Cp"
    CPK = "Cpk"
    STANDARD_DEVIATION = "Standard Deviation"


class SpecComplianceProperties:
    MIN = "Compliance Min"
    MAX = "Compliance Max"
    MEAN = "Compliance Mean"
    MEDIAN = "Compliance Median"
    HEALTH = "Compliance Health"
    CP = "Compliance Cp"
    CPK = "Compliance Cpk"
    STANDARD_DEVIATION = "Compliance Standard Deviation"


class SpecHealth:
    PASS = "Pass"
    FAIL = "Fail"
    INF = "inf"
    NEGATIVE_INF = "-inf"
    NAN = "nan"

MAX_SPECS_PER_UPDATE_REQUEST = 100

### Error messages

In [ ]:
class ErrorMessages:
    MISMATCHING_BASE_UNITS = "Input and output units are having different base unit"

### Configure SystemLink clients

Use the Jupyter HTTP configuration supplied by SystemLink to initialize the specification, Test Monitor, product, and file clients.


In [ ]:
http_configuration = JupyterHttpConfiguration()
spec_client = SpecClient(http_configuration)
test_monitor_client = TestMonitorClient(http_configuration)
product_client = ProductClient(http_configuration)
file_client = FileClient(http_configuration)


### Get product information

Retrieve the product's part number and workspace. The part number scopes measurement results, and the workspace is used when uploading the report.


In [ ]:
product_information = product_client.get_product(product_id)
part_number = product_information.part_number


### Query specifications and measurements

The query functions retrieve selected parametric specs and all matching test steps, following continuation tokens. Measurement selection in this version uses the measurement's `SpecID` and the product's part number; it does not read `measurementMapping`.


In [ ]:
def __generate_spec_ids_filter() -> str:
    return " || ".join(f'specId == "{spec_id}"' for spec_id in spec_ids)


def query_parametric_specs(product_id: str) -> List[Dict]:
    """Query parametric specs for the product, following continuation tokens."""
    request = QuerySpecificationsRequest(
        product_ids=[product_id],
        filter=f'(({__generate_spec_ids_filter()}) && (type == "PARAMETRIC"))',
        take=QUERY_SPECS_TAKE_COUNT,
    )
    specs = []
    response = spec_client.query_specs(request)
    while True:
        specs.extend(
            spec.model_dump(mode="json", by_alias=True, exclude_unset=True)
            for spec in (response.specs or [])
        )
        if not response.continuation_token:
            break
        request.continuation_token = response.continuation_token
        response = spec_client.query_specs(request)
    return specs


def update_specs(updated_specs: List[Dict]):
    """Update specs through the SystemLink Python client."""
    return spec_client.update_specs(UpdateSpecificationsRequest(specs=updated_specs))


def query_steps(part_number: str, spec_id: str) -> List[Dict]:
    """Query measurements for a spec ID within results for the product."""
    request = QueryStepsRequest(
        filter=(
            f'({ApiBody.DATA}.{ApiBody.PARAMETERS}.{ApiBody.ANY}'
            f'({ApiBody.IT}["{ApiBody.SPEC_ID}"] = "{spec_id}"))'
        ),
        result_filter=f'({ApiBody.PART_NUMBER} == "{part_number}")',
        take=QUERY_STEPS_TAKE_COUNT,
        projection=[Projection.DATA, Projection.INPUTS],
    )
    steps = []
    response = test_monitor_client.query_steps(request)
    while True:
        steps.extend(
            step.model_dump(mode="json", by_alias=True, exclude_unset=True)
            for step in (response.steps or [])
        )
        if not response.continuation_token:
            break
        request.continuation_token = response.continuation_token
        response = test_monitor_client.query_steps(request)
    return steps


### Condition mapping and compliance calculations

Step conditions are checked against the spec's condition ranges and discrete values. Compatible units are converted to the spec unit, while empty or non-numeric measurements are excluded. The calculations produce min, max, mean, median, standard deviation, Cp, Cpk, and spec health.


In [ ]:
def __convert_to_decimal(value: str) -> Decimal | None:
    """Converts the value to decimal

    Args:
        value (str): value in string format

    Returns:
        Decimal | None: returns the converted value
    """
    try:
        return Decimal(value)
    except Exception:
        return None


def __convert_to_bool(value: str) -> bool | None:
    """Converts the value to boolean

    Args:
        value (str): value in string format

    Returns:
        bool | None: returns the converted value
    """
    if value in ["True", "true"]:
        return True
    if value in ["False", "false"]:
        return False
    return None


def __convert_to_datatype(value: str) -> bool | Decimal | str:
    """Convert the given string to any one of these types (decimal, bool)

    Args:
        value (str): String value

    Returns:
        bool | Decimal | str: returns the converted value
    """
    bool_value = __convert_to_bool(value=value)
    if bool_value is not None:
        return bool_value

    decimal_value = __convert_to_decimal(value=value)
    if decimal_value is not None:
        return decimal_value

    return value


def get_spec_condition(spec: Dict) -> Dict:
    """
    Get the specification conditions as a dictionary

    Args:
        spec (Dict): Specification data

    Returns:
        Dict: Specification Conditions as a dict
    """
    spec_condition = {}

    for condition in spec[Spec.CONDITIONS]:
        spec_condition[condition[Spec.NAME]] = condition[Spec.VALUE]

    return spec_condition


def extract_condition_name_and_unit(condition_string: str) -> Tuple[str, str | None]:
    """
    Extract the condition name and unit seperately from the condition string

    Args:
        condition_string (str): Condition string which contains the name and unit

    Returns:
        Tuple[str, str | None]: Condition name, Condition Unit
    """
    pattern = r'^(.*?)\((.*?)\)$'
    match = re.match(pattern, condition_string)
    if match:
        condition_name = match.group(1).strip()
        condition_unit = match.group(2).strip()

        return condition_name, condition_unit

    return condition_string.strip(), None


def is_different_base_unit(input_unit: str, output_unit: str) -> bool:
    """Check if both the units have different base unit

    Args:
        input_unit (str): Input unit
        output_unit (str): Output unit

    Returns:
        bool: Returns true if both the units have different base unit, else returns false
    """
    input_base_unit = convert_to_base_unit(input_unit, 1).base_unit
    output_base_unit = convert_to_base_unit(output_unit, 1).base_unit

    if input_base_unit != output_base_unit:
        return True

    return False


def convert_unit(input_unit: str, output_unit: str, value: str) -> Decimal:
    """
    Converts the value from input unit to output unit

    Args:
        input_unit (str): Input unit
        output_unit (str): Ouput unit
        value (str): Value to be converted

    Raises:
        Exception: Input and output units are having different base unit

    Returns:
        Decimal: Value converted from input unit to output unit
    """
    if is_different_base_unit(input_unit, output_unit):
        raise Exception(ErrorMessages.MISMATCHING_BASE_UNITS)
    output_value = convert_to_base_unit(input_unit, value).converted_value
    output_value = convert_from_base_unit(output_unit, output_value)

    return output_value


def get_step_condition_matching_the_spec_condition(step: Dict, spec_condition: Dict) -> Dict:
    """
    Returns the step conditions matching the spec conditions as a dictionary

    Args:
        step (Dict): Step data
        spec_condition (Dict): Spec Conditions dictionary

    Returns:
        Dict: Step conditions matching the spec conditions
    """
    step_condition = {}

    if Step.INPUTS not in step:
        return step_condition
    for step_input in step[Step.INPUTS]:
        step_condition_name, step_condition_unit = extract_condition_name_and_unit(
            step_input[Step.NAME]
        )

        if step_condition_name not in spec_condition:
            continue

        try:
            spec_condition_unit = spec_condition[step_condition_name][Spec.UNIT]
            if spec_condition_unit is None and step_condition_unit:
                continue
        except KeyError:
            spec_condition_unit = None
            if step_condition_unit:
                continue

        if (step_condition_unit == spec_condition_unit) or (
            spec_condition_unit and not step_condition_unit
        ):
            try:
                step_condition[step_condition_name] = (
                    Decimal(str(step_input[Step.VALUE])) if Step.VALUE in step_input else ''
                )
            except InvalidOperation:
                step_condition[step_condition_name] = step_input[Step.VALUE]
        else:
            if is_different_base_unit(step_condition_unit, spec_condition_unit):
                continue
            if Step.VALUE not in step_input or (
                Step.VALUE in step_input and step_input[Step.VALUE] == str()
            ):
                step_condition[step_condition_name] = str()
            else:
                try:
                    step_condition_value = convert_unit(
                        step_condition_unit, spec_condition_unit, str(step_input[Step.VALUE])
                    )
                    step_condition[step_condition_name] = step_condition_value
                except (ValueError, InvalidOperation):
                    step_condition[step_condition_name] = step_input[Step.VALUE]

    return step_condition


def check_step_condition_within_spec_condition_range(
    spec_condition: Dict,
    condition_name: str,
    condition_value: Decimal
) -> bool:
    """
    Checks if the step condition value is within the spec condition range

    Args:
        spec_condition (Dict): Specification condition
        condition_name (str): Condition name
        condition_value (Decimal): Condition value

    Returns:
        bool: Returns true if step condition is within spec condition range , else returns false
    """
    if Spec.RANGE in spec_condition[condition_name] and spec_condition[condition_name][Spec.RANGE]:
        for spec_condition_range in spec_condition[condition_name][Spec.RANGE]:
            min_limit = __convert_to_decimal(str(spec_condition_range[Spec.MIN]))
            max_limit = __convert_to_decimal(str(spec_condition_range[Spec.MAX]))
            if (min_limit is None or condition_value >= min_limit) and (
                max_limit is None or condition_value <= max_limit
            ):
                return True

    return False


def check_step_condition_within_spec_condition_discrete_array(
    spec_condition: Dict,
    condition_name: str,
    condition_value: Decimal | str
) -> bool:
    """
    Check if the step condition value is within the spec condition discrete array

    Args:
        spec_condition (Dict): Specification condition
        condition_name (str): Condition name
        condition_value (Decimal | str): Condition value

    Returns:
        bool: Returns true if step condition value is within the spec condition discrete array else returns false
    """
    if Spec.DISCRETE in spec_condition[condition_name]:
        discrete_values = spec_condition[condition_name][Spec.DISCRETE]
        converted_discrete_values = []
        for discrete_value in discrete_values:
            converted_discrete_values.append(__convert_to_datatype(str(discrete_value)))
        if condition_value in converted_discrete_values:
            return True

    return False


def check_step_condition_within_spec_condition(spec_condition: Dict, step_condition: Dict) -> bool:
    """
    Check if the step condition value is within the spec condition

    Condition mapping rules:
    1. If condition value in measurement data is empty, it is considered as within bounds
    2. If condition value in measurement data is within bounds of either range or discrete array of spec condition, it is considered as within bounds

    Args:
        spec_condition (Dict): Specification condition
        step_condition (Dict): Step condition

    Returns:
        bool: Returns true if  step condition is within the spec condition, else returns false
    """
    condition_within_range = True

    for condition_name, condition_value in step_condition.items():
        try:
            condition_value = Decimal(str(condition_value))
        except InvalidOperation:
            pass

        if not condition_value and isinstance(condition_value, str):
            continue

        condition_within_range = (
            (
                check_step_condition_within_spec_condition_range(
                    spec_condition, condition_name, condition_value
                )
                or check_step_condition_within_spec_condition_discrete_array(
                    spec_condition, condition_name, condition_value
                )
            )
            if isinstance(condition_value, Decimal)
            else check_step_condition_within_spec_condition_discrete_array(
                spec_condition, condition_name, condition_value
            )
        )

        if not condition_within_range:
            return condition_within_range

    return condition_within_range


def fetch_measurement_data_for_spec(
    steps: List,
    spec_id: str,
    spec_condition: Dict,
    spec_unit: str | None
) -> List:
    """
    Returns a list of measurement data associated with the spec after condition mapping

    Measurement data rules:
    1. If Measurement value is empty, it is not considered for compliance
    2. If Measurement value is not a valid number, it is not considered for compliance

    Args:
        steps (List): Steps data
        spec_id (str): Specification ID
        spec_condition (Dict): Specification condition
        spec_unit (str | None): Specification unit

    Returns:
        List: Measurement data
    """
    measurement_data = []

    for step in steps:
        step_condition = get_step_condition_matching_the_spec_condition(step, spec_condition)
        if not check_step_condition_within_spec_condition(spec_condition, step_condition):
            continue
        if Step.DATA not in step or Step.PARAMETERS not in step[Step.DATA]:
            continue
        for parameter in step[Step.DATA][Step.PARAMETERS]:
            step_unit = parameter[Step.UNITS]
            if parameter[Step.UNITS] is str():
                step_unit = None
            if (spec_unit and not step_unit) or (step_unit and not spec_unit):
                continue
            if spec_unit and step_unit and is_different_base_unit(step_unit, spec_unit):
                continue
            if Step.SPEC_ID in parameter and parameter[Step.SPEC_ID] != spec_id:
                continue
            measurement_value = (
                parameter[Step.MEASUREMENT] if Step.MEASUREMENT in parameter else str()
            )
            if measurement_value:
                try:
                    if not spec_unit:
                        measurement_data.append(Decimal(str(measurement_value)))
                    else:
                        measurement_value = convert_from_base_unit(spec_unit, measurement_value)
                        measurement_data.append(measurement_value)
                except (TypeError, ValueError, InvalidOperation):
                    continue

    return measurement_data


def calculate_compliance(measurement_data: pd.DataFrame, parameter: str) -> pd.Series:
    """Calculate compliance for the measurement data

    Args:
        measurement_data (pd.DataFrame): Spec measurement data
        parameter (str): Compliance parameter

    Returns:
        pd.Series: Calculated compliance
    """
    if parameter == ComplianceParameters.MIN:
        return measurement_data.min(skipna=True)
    if parameter == ComplianceParameters.MAX:
        return measurement_data.max(skipna=True)
    if parameter == ComplianceParameters.MEAN:
        return measurement_data.mean(skipna=True)
    if parameter == ComplianceParameters.MEDIAN:
        return measurement_data.median(skipna=True)
    return pd.Series()


def get_spec_health(spec, min_compliance: str, max_compliance: str) -> str | None:
    """Calculate spec health based on min and max compliance

    Args:
        spec (_type_): Specification data
        min_compliance (str): Min compliance value
        max_compliance (str): Max compliance value

    Returns:
        str | None: Returns Pass or Fail, None if there are any errors
    """
    if spec[Spec.UNIT] is not None:
        spec_min = (
            convert_to_base_unit(spec[Spec.UNIT], spec[Spec.LIMIT][Spec.MIN]).converted_value
            if spec[Spec.LIMIT][Spec.MIN] is not None
            else Decimal(SpecHealth.NEGATIVE_INF)
        )
        spec_max = (
            convert_to_base_unit(spec[Spec.UNIT], spec[Spec.LIMIT][Spec.MAX]).converted_value
            if spec[Spec.LIMIT][Spec.MAX] is not None
            else Decimal(SpecHealth.INF)
        )
        min_compliance = convert_to_base_unit(spec[Spec.UNIT], min_compliance).converted_value
        max_compliance = convert_to_base_unit(spec[Spec.UNIT], max_compliance).converted_value
    else:
        spec_min = (
            Decimal(str(spec[Spec.LIMIT][Spec.MIN]))
            if spec[Spec.LIMIT][Spec.MIN] is not None
            else Decimal(SpecHealth.NEGATIVE_INF)
        )
        spec_max = (
            Decimal(str(spec[Spec.LIMIT][Spec.MAX]))
            if spec[Spec.LIMIT][Spec.MAX] is not None
            else Decimal(SpecHealth.INF)
        )
        min_compliance = Decimal(str(min_compliance))
        max_compliance = Decimal(str(max_compliance))

    try:
        if spec_min is Decimal(SpecHealth.NEGATIVE_INF) and spec_max is not None:
            return SpecHealth.PASS if max_compliance <= spec_max else SpecHealth.FAIL
        if spec_max is Decimal(SpecHealth.INF) and spec_min is not None:
            return SpecHealth.PASS if min_compliance <= spec_min else SpecHealth.FAIL
        if spec_min is Decimal(SpecHealth.NEGATIVE_INF) and spec_max is Decimal(SpecHealth.INF):
            return SpecHealth.PASS
        if spec_min <= min_compliance <= spec_max and spec_min <= max_compliance <= spec_max:
            return SpecHealth.PASS
        return SpecHealth.FAIL
    except InvalidOperation:
        return None


def concatenate_dataframes(
    dfs: List[pd.DataFrame],
    reset_index: bool = True,
    axis: int = 1
) -> pd.DataFrame:
    """Concatenate multiple dataframes

    Args:
        dfs (List[pd.DataFrame]): List of dataframes
        reset_index (bool, optional): Reset index. Defaults to True.
        axis (int, optional): Axis to concatenate. Defaults to 1.

    Returns:
        pd.DataFrame: Returns the concatenated dataframe
    """
    if reset_index:
        for df in dfs:
            df.reset_index(drop=True, inplace=True)
    concatenated_df = pd.concat(dfs, axis=axis)

    return concatenated_df


def generate_spec_details_dataframe(spec: Dict) -> pd.DataFrame:
    """Generate spec details dataframe

    Args:
        spec (Dict): Specification data

    Returns:
        pd.DataFrame: Spec detail dataframe
    """
    spec_df = pd.DataFrame(
        {
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.SPEC_ID): [spec[Spec.SPECID]],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.CATEGORY): [spec[Spec.CATEGORY]],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.BLOCK): [spec[Spec.BLOCK]],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.SYMBOL): [spec[Spec.SYMBOL]],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.NAME): [spec[Spec.NAME]],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.MIN): [spec[Spec.LIMIT][Spec.MIN]],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.TYPICAL): [
                spec[Spec.LIMIT][Spec.TYPICAL]
            ],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.MAX): [spec[Spec.LIMIT][Spec.MAX]],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.UNIT): [spec[Spec.UNIT]],
        }
    )

    return spec_df


def generate_spec_conditions_dataframe(spec_condition: Dict) -> pd.DataFrame:
    """Generate spec condition dataframe

    Args:
        spec_condition (Dict): Specification condition dictionary

    Returns:
        pd.DataFrame: Spec condition dataframe
    """
    condition_df = pd.DataFrame()

    for condition_name, condition_data in spec_condition.items():
        condition_header = f"{DataframeHeaders.CONDITION} ({condition_name})"
        if Spec.RANGE in condition_data and len(condition_data[Spec.RANGE]) > 0:
            condition_data = {
                (condition_header, DataframeHeaders.MIN): [
                    range[Spec.MIN] for range in condition_data[Spec.RANGE]
                ],
                (condition_header, DataframeHeaders.STEP): [
                    range[Spec.STEP] for range in condition_data[Spec.RANGE]
                ],
                (condition_header, DataframeHeaders.MAX): [
                    range[Spec.MAX] for range in condition_data[Spec.RANGE]
                ],
                (condition_header, DataframeHeaders.DISCRETE): [
                    condition_data.get(Spec.DISCRETE)
                    if len(condition_data.get(Spec.DISCRETE, [])) > 0
                    else None
                    for _ in condition_data[Spec.RANGE]
                ],
                (condition_header, DataframeHeaders.UNIT): [
                    condition_data.get(Spec.UNIT, None) for _ in condition_data[Spec.RANGE]
                ],
            }
        else:
            condition_data = {
                (condition_header, DataframeHeaders.MIN): [None],
                (condition_header, DataframeHeaders.STEP): [None],
                (condition_header, DataframeHeaders.MAX): [None],
                (condition_header, DataframeHeaders.DISCRETE): [
                    condition_data.get(Spec.DISCRETE)
                    if len(condition_data.get(Spec.DISCRETE, [])) > 0
                    else None
                ],
                (condition_header, DataframeHeaders.UNIT): [
                    condition_data.get(Spec.UNIT, None)
                ],
            }
        spec_condition_df = pd.DataFrame(condition_data)
        condition_df = concatenate_dataframes([condition_df, spec_condition_df])

    return condition_df


def generate_spec_compliance_dataframe(
    spec: Dict,
    rows: int,
    measurement_data: List
) -> pd.DataFrame:
    """Generate spec compliance dataframe

    Args:
        spec (Dict): Specification data
        rows (int): Number of rows to append compliance data
        measurement_data (List): Measurement data

    Returns:
        pd.DataFrame: Spec compliance dataframe
    """
    min_compliance_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.MIN)
    max_compliance_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.MAX)
    mean_compliance_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.MEAN)
    median_compliance_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.MEDIAN)
    spec_health_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.HEALTH)
    spec_cpk_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.CPK)
    spec_ck_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.CP)
    spec_std_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.STANDARD_DEVIATION)
    min_rows = max(1, rows)
    if len(measurement_data) > 0:
        measurement_df = pd.DataFrame({spec[Spec.SPECID]: measurement_data})
        min_compliance = calculate_compliance(measurement_df, ComplianceParameters.MIN)
        max_compliance = calculate_compliance(measurement_df, ComplianceParameters.MAX)
        mean_compliance = calculate_compliance(measurement_df, ComplianceParameters.MEAN)
        median_compliance = calculate_compliance(measurement_df, ComplianceParameters.MEDIAN)
        spec_health = (
            get_spec_health(spec, str(min_compliance.iloc[0]), str(max_compliance.iloc[0]))
            if (str(min_compliance.iloc[0]) != SpecHealth.NAN and str(max_compliance.iloc[0])) != SpecHealth.NAN
            else str()
        )

        std = Decimal(measurement_df.astype('double').std(skipna=True).iloc[0])

        maxima = Decimal(max_compliance.iloc[0])
        minima = Decimal(min_compliance.iloc[0])
        mean = Decimal(mean_compliance.iloc[0])

        is_valid_max = bool(maxima)
        is_valid_min = bool(minima)
        cpk_high = (
            np.nan if not is_valid_max
            else __calculate_cpk_high(maxima, mean, std)
        )
        cpk_low = (
            np.nan if not is_valid_min 
            else __calculate_cpk_low(minima, mean, std)
        )
        cpk = min(cpk_high, cpk_low)
        
        cp = (
            np.nan if not (is_valid_max and is_valid_min)
            else __calculate_cp(maxima, minima, std)
        )

        compliance_data = {
            min_compliance_heading: [min_compliance.iloc[0]] * min_rows,
            max_compliance_heading: [max_compliance.iloc[0]] * min_rows,
            mean_compliance_heading: [mean_compliance.iloc[0]] * min_rows,
            median_compliance_heading: [median_compliance.iloc[0]] * min_rows,
            spec_health_heading: [spec_health] * min_rows,
            spec_std_heading: [std] * min_rows,
            spec_cpk_heading: [cpk] * min_rows,
            spec_ck_heading: [cp] * min_rows
        }
    else:
        empty_compliance_data = [None] * min_rows
        compliance_data = {
            min_compliance_heading: empty_compliance_data,
            max_compliance_heading: empty_compliance_data,
            mean_compliance_heading: empty_compliance_data,
            median_compliance_heading: empty_compliance_data,
            spec_health_heading: empty_compliance_data,
            spec_std_heading: empty_compliance_data,
            spec_cpk_heading: empty_compliance_data,
            spec_ck_heading: empty_compliance_data
        }
    compliance_df = pd.DataFrame(compliance_data)

    return compliance_df


def __calculate_cpk_high(high_limit, mean, std):
    diff = (high_limit) - (mean)
    if std == 0:
        return np.inf
    return diff / (3 * (std))


def __calculate_cpk_low(low_limit, mean, std):
    diff = (mean) - (low_limit)
    if std == 0:
        return np.inf
    return diff / (3 * (std))


def __calculate_cp(high_limit, low_limit, std):
    diff = (high_limit) - (low_limit)
    if std == 0:
        return np.inf
    return diff / (6 * (std))


def generate_spec_dataframe(specs: List, part_number: str) -> Dict:
    """Generate a spec dataframe for each spec category

    Args:
        specs (List): Specifications data
        part_number (str): Part number of the product

    Returns:
        Dict: Dictionary of spec category and dataframe
    """
    spec_data = {}
    spec_compliance_data = {}

    for spec in specs:
        category = spec.get(Spec.CATEGORY, ApiResponse.EMPTY)
        spec_id = spec[Spec.SPECID]
        spec_unit = spec[Spec.UNIT]
        spec_condition = get_spec_condition(spec)
        spec_details_df = generate_spec_details_dataframe(spec)
        spec_conditions_df = generate_spec_conditions_dataframe(spec_condition)
        steps = query_steps(part_number, spec_id)
        measurement_data = fetch_measurement_data_for_spec(
            steps, spec_id, spec_condition, spec_unit
        )
        spec_compliance_df = generate_spec_compliance_dataframe(
            spec, len(spec_conditions_df), measurement_data
        )
        spec_df = concatenate_dataframes([spec_details_df, spec_conditions_df])

        if category not in spec_data:
            spec_data[category] = spec_df
            spec_compliance_data[category] = spec_compliance_df
        else:
            spec_data[category] = concatenate_dataframes([spec_data[category], spec_df], False, 0)
            spec_compliance_data[category] = concatenate_dataframes(
                [spec_compliance_data[category], spec_compliance_df], False, 0
            )

    for category in spec_data:
        spec_data[category] = concatenate_dataframes(
            [spec_data[category], spec_compliance_data[category]]
        )

    return spec_data




### Update compliance properties on specs

Write the calculated metrics to each selected spec's custom properties. Updates are sent in batches, and failed updates are recorded with Scrapbook.


In [ ]:
def __get_batches(list, batch_size):
    for i in range(0, len(list), batch_size):
        yield list[i:i+batch_size]


def __format_compliance(value: Decimal) -> Decimal:
    return '{:.4f}'.format(value)


def __delete_spec_custom_property(spec, property_to_be_deleted) -> None:
    if property_to_be_deleted in spec[Spec.PROPERTIES]:
        del spec[Spec.PROPERTIES][property_to_be_deleted]


def add_compliance_to_spec_custom_properties(specs: List, dataframe_dict: Dict) -> None:
    specs_to_be_updated = []
    for category, dataframe in dataframe_dict.items():
        specIds_df = dataframe[DataframeHeaders.SPEC_DETAILS][DataframeHeaders.SPEC_ID]
        for spec in specs:
            spec_compliance_metrices_df = dataframe[specIds_df == spec[Spec.SPECID]][DataframeHeaders.COMPLIANCE]
            
            if spec_compliance_metrices_df.empty or spec_compliance_metrices_df.iloc[0].empty or (not spec_compliance_metrices_df.iloc[0][ComplianceParameters.HEALTH]):
                __delete_spec_custom_property(spec, SpecComplianceProperties.MIN)
                __delete_spec_custom_property(spec, SpecComplianceProperties.MAX)
                __delete_spec_custom_property(spec, SpecComplianceProperties.MEAN)
                __delete_spec_custom_property(spec, SpecComplianceProperties.MEDIAN)
                __delete_spec_custom_property(spec, SpecComplianceProperties.HEALTH)
                __delete_spec_custom_property(spec, SpecComplianceProperties.STANDARD_DEVIATION)
                __delete_spec_custom_property(spec, SpecComplianceProperties.CPK)
                __delete_spec_custom_property(spec, SpecComplianceProperties.CP)

            else:
                spec[Spec.PROPERTIES][SpecComplianceProperties.MIN] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.MIN]))
                spec[Spec.PROPERTIES][SpecComplianceProperties.MAX] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.MAX]))
                spec[Spec.PROPERTIES][SpecComplianceProperties.MEAN] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.MEAN]))
                spec[Spec.PROPERTIES][SpecComplianceProperties.MEDIAN] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.MEDIAN]))
                spec[Spec.PROPERTIES][SpecComplianceProperties.HEALTH] = str(spec_compliance_metrices_df.iloc[0][ComplianceParameters.HEALTH])
                
                spec[Spec.PROPERTIES][SpecComplianceProperties.STANDARD_DEVIATION] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.STANDARD_DEVIATION]))
                spec[Spec.PROPERTIES][SpecComplianceProperties.CPK] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.CPK]))
                spec[Spec.PROPERTIES][SpecComplianceProperties.CP] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.CP]))
            
            specs_to_be_updated.append(spec)

    for specs_batch in __get_batches(specs_to_be_updated, MAX_SPECS_PER_UPDATE_REQUEST):
        update_specs_response = update_specs(specs_batch)
        if update_specs_response and update_specs_response.failed_specs:
            sb.glue(
                "Error updating spec custom properties with the compliance metrics",
                json.dumps(update_specs_response.model_dump(mode="json", by_alias=True)),
            )
            break


### Generate the Excel report and attach it to the product

Create a worksheet for each spec category, upload the report to the product workspace, and add the uploaded file to the product.


In [ ]:
def upload_file(file_name: str, workspace_id: str) -> str:
    """Upload the compliance report with the product-prefixed file name."""
    file_stream = io.BytesIO(Path(file_name).read_bytes())
    file_stream.name = f"{part_number}-{Path(file_name).name}"
    try:
        return file_client.upload_file(file_stream, workspace=workspace_id)
    finally:
        file_stream.close()


def add_file_id_to_product(product_information, file_id: str):
    """Attach the uploaded report to the product using the Python client."""
    file_ids = list(product_information.file_ids or [])
    file_ids.append(file_id)
    request = UpdateProductRequest(id=product_information.id, file_ids=file_ids)
    return product_client.update_products([request], replace=False)


def write_dataframe_to_excel(dataframe_dict: Dict, file_name: str) -> None:
    """Write each category's compliance dataframe to an Excel sheet."""
    with pd.ExcelWriter(file_name, engine="openpyxl") as writer:
        for category, dataframe in dataframe_dict.items():
            dataframe.reset_index(drop=True, inplace=True)
            dataframe.to_excel(writer, sheet_name=f"Spec_{category}", index=True)


### Fetch specs, calculate compliance, and generate dataframes


In [ ]:
parametric_specs = query_parametric_specs(product_id)
spec_dataframe = generate_spec_dataframe(parametric_specs, part_number)


### Write compliance metrics to spec custom properties


In [ ]:
add_compliance_to_spec_custom_properties(parametric_specs, spec_dataframe)


### Write, upload, and link the Excel report


In [ ]:
pd.set_option("display.float_format", "{:.15f}".format)
write_dataframe_to_excel(spec_dataframe, excel_file_name)
uploaded_file_id = upload_file(
    file_name=excel_file_name,
    workspace_id=product_information.workspace,
)
product_update_response = add_file_id_to_product(
    product_information=product_information,
    file_id=uploaded_file_id,
)


### Store result information with Scrapbook

SystemLink uses Scrapbook to show the report link or update failures in the notebook execution details.


In [ ]:
if product_update_response.failed:
    sb.glue(
        "Error attaching the compliance report to the product",
        json.dumps(product_update_response.model_dump(mode="json", by_alias=True)),
    )
else:
    sb.glue(
        "Compliance Excel report generated and uploaded to the product",
        f'<a href="../../testinsights/products/product/{product_id}/files">Product Files tab</a>',
    )


### Limitations

- This version selects measurements by `SpecID` and product part number. It does not use `measurementMapping`; that selection can be added when the Python client exposes the field.
- Specs with multiple conditions that share a name but use different units may not be represented correctly in the report.
- The inherited minimum-only health calculation currently returns Pass when minimum compliance is at or below the spec minimum, opposite the intended at-or-above rule. Verify that case before using the notebook.


# Next Steps

1. Publish this notebook to SystemLink from the JupyterLab File Browser with the **Specification Analysis** interface.
1. Run it from the Specs grid in the product details page.
1. View the updated custom properties on the spec details page and the compliance report in the product's Files tab.
